# CogAttention — Semantic Stroop

**Track:** Attention — Selective Attention
**Benchmark:** CogAttention v1.0
**Task:** stroop

---

## Methodology

Tests semantic Stroop interference — the model must report what sentences literally say despite containing factual errors, suppressing the correction reflex. Based on the classic Stroop task (Stroop, 1935).

### Cognitive Science Grounding

- **Semantic Stroop Interference** (Stroop, 1935): sentences contain deliberate factual errors; the model must report what the text literally says, not correct it
- Exposes conflict between pre-training priors and in-context instructions — parameterized weights trigger MLP activations that overpower attention heads
- Humans score 0.852 on Stroop resistance; most LLMs score lower

### Difficulty Scaling

Easy: 2 items, obvious factual errors | Medium: 4 items | Hard: 6 items, subtle errors | Expert: 8 items | Frontier: 15 items, multi-hop factual traps

### Scoring

One assertion per Stroop item — checks the model reported the literal (incorrect) statement, not the factually correct answer.

All instances are procedurally generated from a seed with programmatic ground truth.
No static datasets. 7 layers of contamination resistance including canary strings,
zero lexical overlap (Semantic NIAH), and seed-based regeneration.

---

`<!-- COGATTENTION-BENCH-CANARY-EA00EFEB3895 -->`


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2: Imports + Inline Helpers
# CogAttention — Selective Attention
# ══════════════════════════════════════════════════════════════════════

import kaggle_benchmarks as kbench

import json
import re

def extract_answer_block(response):
    for pat in [r"ANSWER:\s*(.*)", r"Answer:\s*(.*)", r"answer:\s*(.*)"]:
        match = re.search(pat, response, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return response.strip()

def extract_numbered_answers(response):
    answer_block = extract_answer_block(response)
    results = {}
    matches = re.findall(
        r"(\d+)\s*[.):\-]\s*(.+?)(?=\n\d+\s*[.):\-]|\Z)",
        answer_block, re.DOTALL,
    )
    for num, val in matches:
        results[num] = val.strip().rstrip(".")
    return results

def extract_list_items(response):
    answer_block = extract_answer_block(response)
    bullets = re.findall(r"[-\u2022]\s*(.+?)(?:\n|$)", answer_block)
    if bullets:
        return [b.strip().rstrip(".") for b in bullets]
    numeric_items = re.findall(
        r'[\$]?\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:\s*(?:\xb0[CF]|mg/L|%|\$))?',
        answer_block,
    )
    if numeric_items and len(numeric_items) >= 2:
        return [x.strip() for x in numeric_items]
    if "," in answer_block:
        items = [x.strip().rstrip(".") for x in answer_block.split(",")]
        return [x for x in items if x]
    lines = [l.strip().rstrip(".") for l in answer_block.split("\n") if l.strip()]
    return lines if lines else ([answer_block] if answer_block else [])

def extract_person_item_pairs(response):
    answer_block = extract_answer_block(response)
    results = {}
    for pat in [
        r"[-\u2022]?\s*(\w+)\s*:\s*(.+?)(?:\n|$)",
        r"[-\u2022]?\s*(\w+)\s+holds?\s+(?:a\s+)?(.+?)(?:\n|$)",
    ]:
        matches = re.findall(pat, answer_block, re.IGNORECASE)
        if matches:
            for name, item in matches:
                results[name.strip()] = item.strip().rstrip(".")
            break
    return results

def fuzzy_value_match(predicted, gold):
    pred_clean = re.sub(r"\s+", " ", predicted.strip().lower())
    gold_clean = re.sub(r"\s+", " ", gold.strip().lower())
    if pred_clean == gold_clean:
        return True
    if gold_clean in pred_clean:
        return True
    try:
        pred_num = float(re.sub(r"[,$%\xb0]", "", predicted))
        gold_num = float(re.sub(r"[,$%\xb0]", "", gold))
        return pred_num == gold_num
    except (ValueError, TypeError):
        pass
    return False

def _escape_for_regex(s):
    return re.escape(s).replace(r"\ ", r"\s+")


def run_assertions_stroop(response, gold, kbench):
    for idx_str, gold_val in gold["answers"].items():
        pattern = rf"(?i){re.escape(idx_str)}\s*[.):\-]\s*.*{_escape_for_regex(gold_val)}"
        trap = gold["traps"].get(idx_str)
        trap_note = f" (trap: '{trap}')" if trap else ""
        kbench.assertions.assert_contains_regex(
            pattern, response,
            expectation=f"Item {idx_str} should be '{gold_val}'{trap_note}"
        )


print("CogAttention helpers loaded")
print(f"Task types: ['stroop']")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3: Task Definitions + Embedded Dataset
# ══════════════════════════════════════════════════════════════════════


@kbench.task(name="cogattention_stroop")
def cogattention_stroop(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention stroop task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_stroop(response, gold, kbench)


# ── Embedded dataset ──────────────────────────────────────────────────
DATASET = json.loads(r'''
[
 {
  "task_id": "stroop_easy_000",
  "task_type": "stroop",
  "difficulty": "Easy",
  "prompt": "IMPORTANT: For each sentence below, answer the question about what the sentence LITERALLY SAYS. Do NOT correct any factual errors — report exactly what is stated in the sentence, even if it is wrong.\n\n1. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"The French Revolution occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n3. Sentence: \"The signing of the Magna Carta occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Italy\", \"2\": \"1955\", \"3\": \"1776\"}, \"traps\": {\"1\": \"France\", \"2\": \"the actual year\", \"3\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_easy_001",
  "task_type": "stroop",
  "difficulty": "Easy",
  "prompt": "IMPORTANT: For each sentence below, answer the question about what the sentence LITERALLY SAYS. Do NOT correct any factual errors — report exactly what is stated in the sentence, even if it is wrong.\n\n1. Sentence: \"The French Revolution occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n2. Sentence: \"The Volga is the longest river in Europe\"\n   Question: What river is mentioned?\n\n3. Sentence: \"The signing of the Magna Carta occurred in 1801\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1955\", \"2\": \"Volga\", \"3\": \"1801\"}, \"traps\": {\"1\": \"the actual year\", \"2\": \"the actual longest river\", \"3\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_easy_002",
  "task_type": "stroop",
  "difficulty": "Easy",
  "prompt": "IMPORTANT: For each sentence below, answer the question about what the sentence LITERALLY SAYS. Do NOT correct any factual errors — report exactly what is stated in the sentence, even if it is wrong.\n\n1. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"The Moon landing occurred in 1848\"\n   Question: What year is mentioned in the sentence?\n\n3. Sentence: \"The Seine is the longest river in Africa\"\n   Question: What river is mentioned?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Brazil\", \"2\": \"1848\", \"3\": \"Seine\"}, \"traps\": {\"1\": \"France\", \"2\": \"the actual year\", \"3\": \"the actual longest river\"}}"
 },
 {
  "task_id": "stroop_easy_003",
  "task_type": "stroop",
  "difficulty": "Easy",
  "prompt": "IMPORTANT: For each sentence below, answer the question about what the sentence LITERALLY SAYS. Do NOT correct any factual errors — report exactly what is stated in the sentence, even if it is wrong.\n\n1. Sentence: \"The Fall of the Berlin Wall occurred in 1997\"\n   Question: What year is mentioned in the sentence?\n\n2. Sentence: \"The Volga is the longest river in South America\"\n   Question: What river is mentioned?\n\n3. Sentence: \"The French Revolution occurred in 1997\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1997\", \"2\": \"Volga\", \"3\": \"1997\"}, \"traps\": {\"1\": \"the actual year\", \"2\": \"the actual longest river\", \"3\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_easy_004",
  "task_type": "stroop",
  "difficulty": "Easy",
  "prompt": "IMPORTANT: For each sentence below, answer the question about what the sentence LITERALLY SAYS. Do NOT correct any factual errors — report exactly what is stated in the sentence, even if it is wrong.\n\n1. Sentence: \"The Moon landing occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\n2. Sentence: \"According to the report, 65 + 28 = 97\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"According to the report, 45 + 34 = 85\"\n   Question: What number does the report state as the sum?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1776\", \"2\": \"97\", \"3\": \"85\"}, \"traps\": {\"1\": \"the actual year\", \"2\": \"93\", \"3\": \"79\"}}"
 },
 {
  "task_id": "stroop_easy_005",
  "task_type": "stroop",
  "difficulty": "Easy",
  "prompt": "IMPORTANT: For each sentence below, answer the question about what the sentence LITERALLY SAYS. Do NOT correct any factual errors — report exactly what is stated in the sentence, even if it is wrong.\n\n1. Sentence: \"According to the report, 36 + 22 = 50\"\n   Question: What number does the report state as the sum?\n\n2. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n3. Sentence: \"The Moon landing occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"50\", \"2\": \"Italy\", \"3\": \"1955\"}, \"traps\": {\"1\": \"58\", \"2\": \"France\", \"3\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_easy_006",
  "task_type": "stroop",
  "difficulty": "Easy",
  "prompt": "IMPORTANT: For each sentence below, answer the question about what the sentence LITERALLY SAYS. Do NOT correct any factual errors — report exactly what is stated in the sentence, even if it is wrong.\n\n1. Sentence: \"The signing of the Magna Carta occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"The Fall of the Berlin Wall occurred in 1997\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1955\", \"2\": \"92\", \"3\": \"1997\"}, \"traps\": {\"1\": \"the actual year\", \"2\": \"100\", \"3\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_easy_007",
  "task_type": "stroop",
  "difficulty": "Easy",
  "prompt": "IMPORTANT: For each sentence below, answer the question about what the sentence LITERALLY SAYS. Do NOT correct any factual errors — report exactly what is stated in the sentence, even if it is wrong.\n\n1. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"According to the report, 31 + 28 = 51\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Australia\", \"2\": \"51\", \"3\": \"120\"}, \"traps\": {\"1\": \"France\", \"2\": \"59\", \"3\": \"100\"}}"
 },
 {
  "task_id": "stroop_medium_008",
  "task_type": "stroop",
  "difficulty": "Medium",
  "prompt": "Answer each question based ONLY on what the sentence states. Do not correct errors in the sentences.\n\n1. Sentence: \"According to the report, 39 + 79 = 121\"\n   Question: What number does the report state as the sum?\n\n2. Sentence: \"The Moon landing occurred in 1801\"\n   Question: What year is mentioned in the sentence?\n\n3. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"121\", \"2\": \"1801\", \"3\": \"Spain\", \"4\": \"Australia\"}, \"traps\": {\"1\": \"118\", \"2\": \"the actual year\", \"3\": \"France\", \"4\": \"France\"}}"
 },
 {
  "task_id": "stroop_medium_009",
  "task_type": "stroop",
  "difficulty": "Medium",
  "prompt": "Answer each question based ONLY on what the sentence states. Do not correct errors in the sentences.\n\n1. Sentence: \"According to the report, 69 + 56 = 128\"\n   Question: What number does the report state as the sum?\n\n2. Sentence: \"According to the report, 80 + 87 = 171\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"The Volga is the longest river in Europe\"\n   Question: What river is mentioned?\n\n4. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"128\", \"2\": \"171\", \"3\": \"Volga\", \"4\": \"Italy\"}, \"traps\": {\"1\": \"125\", \"2\": \"167\", \"3\": \"the actual longest river\", \"4\": \"France\"}}"
 },
 {
  "task_id": "stroop_medium_010",
  "task_type": "stroop",
  "difficulty": "Medium",
  "prompt": "Answer each question based ONLY on what the sentence states. Do not correct errors in the sentences.\n\n1. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"According to the report, 59 + 83 = 139\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"The Moon landing occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"75\", \"2\": \"75\", \"3\": \"139\", \"4\": \"1776\"}, \"traps\": {\"1\": \"100\", \"2\": \"100\", \"3\": \"142\", \"4\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_medium_011",
  "task_type": "stroop",
  "difficulty": "Medium",
  "prompt": "Answer each question based ONLY on what the sentence states. Do not correct errors in the sentences.\n\n1. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"The Moon landing occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\n3. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Australia\", \"2\": \"1776\", \"3\": \"Brazil\", \"4\": \"Brazil\"}, \"traps\": {\"1\": \"France\", \"2\": \"the actual year\", \"3\": \"France\", \"4\": \"France\"}}"
 },
 {
  "task_id": "stroop_medium_012",
  "task_type": "stroop",
  "difficulty": "Medium",
  "prompt": "Answer each question based ONLY on what the sentence states. Do not correct errors in the sentences.\n\n1. Sentence: \"The Fall of the Berlin Wall occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 110 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"Water boils at 110 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n4. Sentence: \"Paris is the capital of Japan\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1955\", \"2\": \"110\", \"3\": \"110\", \"4\": \"Japan\"}, \"traps\": {\"1\": \"the actual year\", \"2\": \"100\", \"3\": \"100\", \"4\": \"France\"}}"
 },
 {
  "task_id": "stroop_medium_013",
  "task_type": "stroop",
  "difficulty": "Medium",
  "prompt": "Answer each question based ONLY on what the sentence states. Do not correct errors in the sentences.\n\n1. Sentence: \"The Danube is the longest river in South America\"\n   Question: What river is mentioned?\n\n2. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"Water boils at 110 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n4. Sentence: \"According to the report, 70 + 81 = 166\"\n   Question: What number does the report state as the sum?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Danube\", \"2\": \"85\", \"3\": \"110\", \"4\": \"166\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"100\", \"3\": \"100\", \"4\": \"151\"}}"
 },
 {
  "task_id": "stroop_medium_014",
  "task_type": "stroop",
  "difficulty": "Medium",
  "prompt": "Answer each question based ONLY on what the sentence states. Do not correct errors in the sentences.\n\n1. Sentence: \"The Danube is the longest river in South America\"\n   Question: What river is mentioned?\n\n2. Sentence: \"The signing of the Magna Carta occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\n3. Sentence: \"According to the report, 33 + 49 = 79\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"The Moon landing occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Danube\", \"2\": \"1302\", \"3\": \"79\", \"4\": \"1776\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"the actual year\", \"3\": \"82\", \"4\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_medium_015",
  "task_type": "stroop",
  "difficulty": "Medium",
  "prompt": "Answer each question based ONLY on what the sentence states. Do not correct errors in the sentences.\n\n1. Sentence: \"Paris is the capital of Japan\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Japan\", \"2\": \"85\", \"3\": \"Spain\", \"4\": \"75\"}, \"traps\": {\"1\": \"France\", \"2\": \"100\", \"3\": \"France\", \"4\": \"100\"}}"
 },
 {
  "task_id": "stroop_hard_016",
  "task_type": "stroop",
  "difficulty": "Hard",
  "prompt": "For each item, answer the question about the sentence as written.\n\n1. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"According to the report, 88 + 48 = 132\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"The Danube is the longest river in Africa\"\n   Question: What river is mentioned?\n\n4. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n5. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n6. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Germany\", \"2\": \"132\", \"3\": \"Danube\", \"4\": \"Brazil\", \"5\": \"Spain\", \"6\": \"Australia\"}, \"traps\": {\"1\": \"France\", \"2\": \"136\", \"3\": \"the actual longest river\", \"4\": \"France\", \"5\": \"France\", \"6\": \"France\"}}"
 },
 {
  "task_id": "stroop_hard_017",
  "task_type": "stroop",
  "difficulty": "Hard",
  "prompt": "For each item, answer the question about the sentence as written.\n\n1. Sentence: \"According to the report, 59 + 89 = 151\"\n   Question: What number does the report state as the sum?\n\n2. Sentence: \"According to the report, 38 + 33 = 56\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n4. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n5. Sentence: \"According to the report, 72 + 80 = 156\"\n   Question: What number does the report state as the sum?\n\n6. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"151\", \"2\": \"56\", \"3\": \"85\", \"4\": \"Italy\", \"5\": \"156\", \"6\": \"92\"}, \"traps\": {\"1\": \"148\", \"2\": \"71\", \"3\": \"100\", \"4\": \"France\", \"5\": \"152\", \"6\": \"100\"}}"
 },
 {
  "task_id": "stroop_hard_018",
  "task_type": "stroop",
  "difficulty": "Hard",
  "prompt": "For each item, answer the question about the sentence as written.\n\n1. Sentence: \"The Thames is the longest river in Africa\"\n   Question: What river is mentioned?\n\n2. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n3. Sentence: \"According to the report, 71 + 53 = 114\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n5. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n6. Sentence: \"According to the report, 87 + 64 = 156\"\n   Question: What number does the report state as the sum?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Thames\", \"2\": \"Australia\", \"3\": \"114\", \"4\": \"Australia\", \"5\": \"Italy\", \"6\": \"156\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"France\", \"3\": \"124\", \"4\": \"France\", \"5\": \"France\", \"6\": \"151\"}}"
 },
 {
  "task_id": "stroop_hard_019",
  "task_type": "stroop",
  "difficulty": "Hard",
  "prompt": "For each item, answer the question about the sentence as written.\n\n1. Sentence: \"Water boils at 110 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 150 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"The Danube is the longest river in Asia\"\n   Question: What river is mentioned?\n\n4. Sentence: \"The Moon landing occurred in 1801\"\n   Question: What year is mentioned in the sentence?\n\n5. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n6. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"110\", \"2\": \"150\", \"3\": \"Danube\", \"4\": \"1801\", \"5\": \"Italy\", \"6\": \"Germany\"}, \"traps\": {\"1\": \"100\", \"2\": \"100\", \"3\": \"the actual longest river\", \"4\": \"the actual year\", \"5\": \"France\", \"6\": \"France\"}}"
 },
 {
  "task_id": "stroop_hard_020",
  "task_type": "stroop",
  "difficulty": "Hard",
  "prompt": "For each item, answer the question about the sentence as written.\n\n1. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"The Seine is the longest river in Africa\"\n   Question: What river is mentioned?\n\n3. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n5. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n6. Sentence: \"The Seine is the longest river in South America\"\n   Question: What river is mentioned?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Australia\", \"2\": \"Seine\", \"3\": \"Germany\", \"4\": \"75\", \"5\": \"92\", \"6\": \"Seine\"}, \"traps\": {\"1\": \"France\", \"2\": \"the actual longest river\", \"3\": \"France\", \"4\": \"100\", \"5\": \"100\", \"6\": \"the actual longest river\"}}"
 },
 {
  "task_id": "stroop_hard_021",
  "task_type": "stroop",
  "difficulty": "Hard",
  "prompt": "For each item, answer the question about the sentence as written.\n\n1. Sentence: \"The Rhine is the longest river in Africa\"\n   Question: What river is mentioned?\n\n2. Sentence: \"The French Revolution occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n3. Sentence: \"The Volga is the longest river in Africa\"\n   Question: What river is mentioned?\n\n4. Sentence: \"The Thames is the longest river in Europe\"\n   Question: What river is mentioned?\n\n5. Sentence: \"The Thames is the longest river in Europe\"\n   Question: What river is mentioned?\n\n6. Sentence: \"According to the report, 15 + 54 = 71\"\n   Question: What number does the report state as the sum?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Rhine\", \"2\": \"1955\", \"3\": \"Volga\", \"4\": \"Thames\", \"5\": \"Thames\", \"6\": \"71\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"the actual year\", \"3\": \"the actual longest river\", \"4\": \"the actual longest river\", \"5\": \"the actual longest river\", \"6\": \"69\"}}"
 },
 {
  "task_id": "stroop_hard_022",
  "task_type": "stroop",
  "difficulty": "Hard",
  "prompt": "For each item, answer the question about the sentence as written.\n\n1. Sentence: \"The French Revolution occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n2. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n3. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n4. Sentence: \"The Fall of the Berlin Wall occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\n5. Sentence: \"According to the report, 14 + 89 = 107\"\n   Question: What number does the report state as the sum?\n\n6. Sentence: \"According to the report, 19 + 26 = 50\"\n   Question: What number does the report state as the sum?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1955\", \"2\": \"Italy\", \"3\": \"75\", \"4\": \"1776\", \"5\": \"107\", \"6\": \"50\"}, \"traps\": {\"1\": \"the actual year\", \"2\": \"France\", \"3\": \"100\", \"4\": \"the actual year\", \"5\": \"103\", \"6\": \"45\"}}"
 },
 {
  "task_id": "stroop_hard_023",
  "task_type": "stroop",
  "difficulty": "Hard",
  "prompt": "For each item, answer the question about the sentence as written.\n\n1. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n2. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\n3. Sentence: \"According to the report, 62 + 73 = 126\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"Water boils at 150 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n5. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n6. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"85\", \"2\": \"Germany\", \"3\": \"126\", \"4\": \"150\", \"5\": \"Spain\", \"6\": \"85\"}, \"traps\": {\"1\": \"100\", \"2\": \"France\", \"3\": \"135\", \"4\": \"100\", \"5\": \"France\", \"6\": \"100\"}}"
 },
 {
  "task_id": "stroop_expert_024",
  "task_type": "stroop",
  "difficulty": "Expert",
  "prompt": "Answer each question.\n\n1. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"According to the report, 73 + 10 = 80\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"The Rhine is the longest river in Europe\"\n   Question: What river is mentioned?\n\n5. Sentence: \"According to the report, 81 + 89 = 167\"\n   Question: What number does the report state as the sum?\n\n6. Sentence: \"The Fall of the Berlin Wall occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n7. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n8. Sentence: \"According to the report, 41 + 37 = 73\"\n   Question: What number does the report state as the sum?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Germany\", \"2\": \"80\", \"3\": \"Italy\", \"4\": \"Rhine\", \"5\": \"167\", \"6\": \"1955\", \"7\": \"92\", \"8\": \"73\"}, \"traps\": {\"1\": \"France\", \"2\": \"83\", \"3\": \"France\", \"4\": \"the actual longest river\", \"5\": \"170\", \"6\": \"the actual year\", \"7\": \"100\", \"8\": \"78\"}}"
 },
 {
  "task_id": "stroop_expert_025",
  "task_type": "stroop",
  "difficulty": "Expert",
  "prompt": "Answer each question.\n\n1. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n4. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n5. Sentence: \"According to the report, 80 + 65 = 135\"\n   Question: What number does the report state as the sum?\n\n6. Sentence: \"According to the report, 88 + 77 = 162\"\n   Question: What number does the report state as the sum?\n\n7. Sentence: \"According to the report, 54 + 18 = 57\"\n   Question: What number does the report state as the sum?\n\n8. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"92\", \"2\": \"120\", \"3\": \"92\", \"4\": \"75\", \"5\": \"135\", \"6\": \"162\", \"7\": \"57\", \"8\": \"Brazil\"}, \"traps\": {\"1\": \"100\", \"2\": \"100\", \"3\": \"100\", \"4\": \"100\", \"5\": \"145\", \"6\": \"165\", \"7\": \"72\", \"8\": \"France\"}}"
 },
 {
  "task_id": "stroop_expert_026",
  "task_type": "stroop",
  "difficulty": "Expert",
  "prompt": "Answer each question.\n\n1. Sentence: \"The French Revolution occurred in 1848\"\n   Question: What year is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"The Seine is the longest river in Africa\"\n   Question: What river is mentioned?\n\n5. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n6. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n7. Sentence: \"The Seine is the longest river in Asia\"\n   Question: What river is mentioned?\n\n8. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1848\", \"2\": \"92\", \"3\": \"Australia\", \"4\": \"Seine\", \"5\": \"120\", \"6\": \"85\", \"7\": \"Seine\", \"8\": \"Brazil\"}, \"traps\": {\"1\": \"the actual year\", \"2\": \"100\", \"3\": \"France\", \"4\": \"the actual longest river\", \"5\": \"100\", \"6\": \"100\", \"7\": \"the actual longest river\", \"8\": \"France\"}}"
 },
 {
  "task_id": "stroop_expert_027",
  "task_type": "stroop",
  "difficulty": "Expert",
  "prompt": "Answer each question.\n\n1. Sentence: \"The Thames is the longest river in South America\"\n   Question: What river is mentioned?\n\n2. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"The Rhine is the longest river in Africa\"\n   Question: What river is mentioned?\n\n4. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n5. Sentence: \"According to the report, 32 + 31 = 51\"\n   Question: What number does the report state as the sum?\n\n6. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n7. Sentence: \"The Moon landing occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\n8. Sentence: \"The Danube is the longest river in Asia\"\n   Question: What river is mentioned?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Thames\", \"2\": \"120\", \"3\": \"Rhine\", \"4\": \"120\", \"5\": \"51\", \"6\": \"75\", \"7\": \"1776\", \"8\": \"Danube\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"100\", \"3\": \"the actual longest river\", \"4\": \"100\", \"5\": \"63\", \"6\": \"100\", \"7\": \"the actual year\", \"8\": \"the actual longest river\"}}"
 },
 {
  "task_id": "stroop_expert_028",
  "task_type": "stroop",
  "difficulty": "Expert",
  "prompt": "Answer each question.\n\n1. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n2. Sentence: \"According to the report, 42 + 71 = 110\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"Water boils at 150 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n4. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n5. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\n6. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n7. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n8. Sentence: \"The signing of the Magna Carta occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"120\", \"2\": \"110\", \"3\": \"150\", \"4\": \"Brazil\", \"5\": \"Germany\", \"6\": \"Spain\", \"7\": \"92\", \"8\": \"1302\"}, \"traps\": {\"1\": \"100\", \"2\": \"113\", \"3\": \"100\", \"4\": \"France\", \"5\": \"France\", \"6\": \"France\", \"7\": \"100\", \"8\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_expert_029",
  "task_type": "stroop",
  "difficulty": "Expert",
  "prompt": "Answer each question.\n\n1. Sentence: \"The Volga is the longest river in South America\"\n   Question: What river is mentioned?\n\n2. Sentence: \"The Moon landing occurred in 1801\"\n   Question: What year is mentioned in the sentence?\n\n3. Sentence: \"According to the report, 71 + 22 = 95\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"The Fall of the Berlin Wall occurred in 1997\"\n   Question: What year is mentioned in the sentence?\n\n5. Sentence: \"The Moon landing occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\n6. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n7. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n8. Sentence: \"According to the report, 89 + 28 = 120\"\n   Question: What number does the report state as the sum?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Volga\", \"2\": \"1801\", \"3\": \"95\", \"4\": \"1997\", \"5\": \"1302\", \"6\": \"Brazil\", \"7\": \"Brazil\", \"8\": \"120\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"the actual year\", \"3\": \"93\", \"4\": \"the actual year\", \"5\": \"the actual year\", \"6\": \"France\", \"7\": \"France\", \"8\": \"117\"}}"
 },
 {
  "task_id": "stroop_expert_030",
  "task_type": "stroop",
  "difficulty": "Expert",
  "prompt": "Answer each question.\n\n1. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n2. Sentence: \"According to the report, 24 + 10 = 19\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"The Thames is the longest river in Asia\"\n   Question: What river is mentioned?\n\n5. Sentence: \"The Moon landing occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\n6. Sentence: \"The French Revolution occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\n7. Sentence: \"Paris is the capital of Japan\"\n   Question: What country is mentioned in the sentence?\n\n8. Sentence: \"The Moon landing occurred in 1801\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"92\", \"2\": \"19\", \"3\": \"Germany\", \"4\": \"Thames\", \"5\": \"1776\", \"6\": \"1302\", \"7\": \"Japan\", \"8\": \"1801\"}, \"traps\": {\"1\": \"100\", \"2\": \"34\", \"3\": \"France\", \"4\": \"the actual longest river\", \"5\": \"the actual year\", \"6\": \"the actual year\", \"7\": \"France\", \"8\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_expert_031",
  "task_type": "stroop",
  "difficulty": "Expert",
  "prompt": "Answer each question.\n\n1. Sentence: \"The Danube is the longest river in South America\"\n   Question: What river is mentioned?\n\n2. Sentence: \"The Volga is the longest river in Africa\"\n   Question: What river is mentioned?\n\n3. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n5. Sentence: \"The Danube is the longest river in Asia\"\n   Question: What river is mentioned?\n\n6. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n7. Sentence: \"According to the report, 78 + 56 = 126\"\n   Question: What number does the report state as the sum?\n\n8. Sentence: \"The Volga is the longest river in South America\"\n   Question: What river is mentioned?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Danube\", \"2\": \"Volga\", \"3\": \"Spain\", \"4\": \"120\", \"5\": \"Danube\", \"6\": \"85\", \"7\": \"126\", \"8\": \"Volga\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"the actual longest river\", \"3\": \"France\", \"4\": \"100\", \"5\": \"the actual longest river\", \"6\": \"100\", \"7\": \"134\", \"8\": \"the actual longest river\"}}"
 },
 {
  "task_id": "stroop_frontier_032",
  "task_type": "stroop",
  "difficulty": "Frontier",
  "prompt": "You will read a series of statements. Each one contains a factual claim. Some claims are accurate, others are not. For each item, a question is posed. Note: the questions are about the literal content of the sentences, not about real-world correctness. Keep this distinction in mind as you proceed.\n\n1. Sentence: \"Water boils at 150 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"According to the report, 29 + 45 = 77\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"The Moon landing occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\n5. Sentence: \"The Volga is the longest river in Europe\"\n   Question: What river is mentioned?\n\n6. Sentence: \"The Moon landing occurred in 1848\"\n   Question: What year is mentioned in the sentence?\n\n7. Sentence: \"The Danube is the longest river in Europe\"\n   Question: What river is mentioned?\n\n8. Sentence: \"The signing of the Magna Carta occurred in 1848\"\n   Question: What year is mentioned in the sentence?\n\n9. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n10. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n11. Sentence: \"The Seine is the longest river in Europe\"\n   Question: What river is mentioned?\n\n12. Sentence: \"The Volga is the longest river in Asia\"\n   Question: What river is mentioned?\n\n13. Sentence: \"The Thames is the longest river in Europe\"\n   Question: What river is mentioned?\n\n14. Sentence: \"The Volga is the longest river in Asia\"\n   Question: What river is mentioned?\n\n15. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]\n9. [your answer]\n10. [your answer]\n11. [your answer]\n12. [your answer]\n13. [your answer]\n14. [your answer]\n15. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"150\", \"2\": \"85\", \"3\": \"77\", \"4\": \"1302\", \"5\": \"Volga\", \"6\": \"1848\", \"7\": \"Danube\", \"8\": \"1848\", \"9\": \"75\", \"10\": \"Australia\", \"11\": \"Seine\", \"12\": \"Volga\", \"13\": \"Thames\", \"14\": \"Volga\", \"15\": \"75\"}, \"traps\": {\"1\": \"100\", \"2\": \"100\", \"3\": \"74\", \"4\": \"the actual year\", \"5\": \"the actual longest river\", \"6\": \"the actual year\", \"7\": \"the actual longest river\", \"8\": \"the actual year\", \"9\": \"100\", \"10\": \"France\", \"11\": \"the actual longest river\", \"12\": \"the actual longest river\", \"13\": \"the actual longest river\", \"14\": \"the actual longest river\", \"15\": \"100\"}}"
 },
 {
  "task_id": "stroop_frontier_033",
  "task_type": "stroop",
  "difficulty": "Frontier",
  "prompt": "You will read a series of statements. Each one contains a factual claim. Some claims are accurate, others are not. For each item, a question is posed. Note: the questions are about the literal content of the sentences, not about real-world correctness. Keep this distinction in mind as you proceed.\n\n1. Sentence: \"The Volga is the longest river in Europe\"\n   Question: What river is mentioned?\n\n2. Sentence: \"The Rhine is the longest river in Europe\"\n   Question: What river is mentioned?\n\n3. Sentence: \"According to the report, 20 + 34 = 46\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"According to the report, 42 + 34 = 91\"\n   Question: What number does the report state as the sum?\n\n5. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n6. Sentence: \"According to the report, 29 + 55 = 90\"\n   Question: What number does the report state as the sum?\n\n7. Sentence: \"Paris is the capital of Japan\"\n   Question: What country is mentioned in the sentence?\n\n8. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n9. Sentence: \"According to the report, 36 + 98 = 142\"\n   Question: What number does the report state as the sum?\n\n10. Sentence: \"According to the report, 39 + 43 = 81\"\n   Question: What number does the report state as the sum?\n\n11. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n12. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n13. Sentence: \"According to the report, 65 + 95 = 156\"\n   Question: What number does the report state as the sum?\n\n14. Sentence: \"The signing of the Magna Carta occurred in 1997\"\n   Question: What year is mentioned in the sentence?\n\n15. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]\n9. [your answer]\n10. [your answer]\n11. [your answer]\n12. [your answer]\n13. [your answer]\n14. [your answer]\n15. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Volga\", \"2\": \"Rhine\", \"3\": \"46\", \"4\": \"91\", \"5\": \"120\", \"6\": \"90\", \"7\": \"Japan\", \"8\": \"92\", \"9\": \"142\", \"10\": \"81\", \"11\": \"Brazil\", \"12\": \"85\", \"13\": \"156\", \"14\": \"1997\", \"15\": \"Italy\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"the actual longest river\", \"3\": \"54\", \"4\": \"76\", \"5\": \"100\", \"6\": \"84\", \"7\": \"France\", \"8\": \"100\", \"9\": \"134\", \"10\": \"82\", \"11\": \"France\", \"12\": \"100\", \"13\": \"160\", \"14\": \"the actual year\", \"15\": \"France\"}}"
 },
 {
  "task_id": "stroop_frontier_034",
  "task_type": "stroop",
  "difficulty": "Frontier",
  "prompt": "You will read a series of statements. Each one contains a factual claim. Some claims are accurate, others are not. For each item, a question is posed. Note: the questions are about the literal content of the sentences, not about real-world correctness. Keep this distinction in mind as you proceed.\n\n1. Sentence: \"According to the report, 18 + 45 = 73\"\n   Question: What number does the report state as the sum?\n\n2. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n3. Sentence: \"The Fall of the Berlin Wall occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n4. Sentence: \"According to the report, 61 + 92 = 149\"\n   Question: What number does the report state as the sum?\n\n5. Sentence: \"Paris is the capital of Japan\"\n   Question: What country is mentioned in the sentence?\n\n6. Sentence: \"According to the report, 98 + 24 = 124\"\n   Question: What number does the report state as the sum?\n\n7. Sentence: \"The Moon landing occurred in 1801\"\n   Question: What year is mentioned in the sentence?\n\n8. Sentence: \"According to the report, 54 + 94 = 144\"\n   Question: What number does the report state as the sum?\n\n9. Sentence: \"The Thames is the longest river in Asia\"\n   Question: What river is mentioned?\n\n10. Sentence: \"The Fall of the Berlin Wall occurred in 1997\"\n   Question: What year is mentioned in the sentence?\n\n11. Sentence: \"The Moon landing occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\n12. Sentence: \"The Seine is the longest river in Africa\"\n   Question: What river is mentioned?\n\n13. Sentence: \"The Rhine is the longest river in Europe\"\n   Question: What river is mentioned?\n\n14. Sentence: \"According to the report, 86 + 51 = 134\"\n   Question: What number does the report state as the sum?\n\n15. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]\n9. [your answer]\n10. [your answer]\n11. [your answer]\n12. [your answer]\n13. [your answer]\n14. [your answer]\n15. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"73\", \"2\": \"Italy\", \"3\": \"1955\", \"4\": \"149\", \"5\": \"Japan\", \"6\": \"124\", \"7\": \"1801\", \"8\": \"144\", \"9\": \"Thames\", \"10\": \"1997\", \"11\": \"1302\", \"12\": \"Seine\", \"13\": \"Rhine\", \"14\": \"134\", \"15\": \"120\"}, \"traps\": {\"1\": \"63\", \"2\": \"France\", \"3\": \"the actual year\", \"4\": \"153\", \"5\": \"France\", \"6\": \"122\", \"7\": \"the actual year\", \"8\": \"148\", \"9\": \"the actual longest river\", \"10\": \"the actual year\", \"11\": \"the actual year\", \"12\": \"the actual longest river\", \"13\": \"the actual longest river\", \"14\": \"137\", \"15\": \"100\"}}"
 },
 {
  "task_id": "stroop_frontier_035",
  "task_type": "stroop",
  "difficulty": "Frontier",
  "prompt": "You will read a series of statements. Each one contains a factual claim. Some claims are accurate, others are not. For each item, a question is posed. Note: the questions are about the literal content of the sentences, not about real-world correctness. Keep this distinction in mind as you proceed.\n\n1. Sentence: \"The signing of the Magna Carta occurred in 1955\"\n   Question: What year is mentioned in the sentence?\n\n2. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n3. Sentence: \"The Rhine is the longest river in Asia\"\n   Question: What river is mentioned?\n\n4. Sentence: \"Water boils at 150 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n5. Sentence: \"The Thames is the longest river in Europe\"\n   Question: What river is mentioned?\n\n6. Sentence: \"Paris is the capital of Japan\"\n   Question: What country is mentioned in the sentence?\n\n7. Sentence: \"The signing of the Magna Carta occurred in 1848\"\n   Question: What year is mentioned in the sentence?\n\n8. Sentence: \"Paris is the capital of Japan\"\n   Question: What country is mentioned in the sentence?\n\n9. Sentence: \"The Danube is the longest river in South America\"\n   Question: What river is mentioned?\n\n10. Sentence: \"According to the report, 44 + 49 = 99\"\n   Question: What number does the report state as the sum?\n\n11. Sentence: \"The Fall of the Berlin Wall occurred in 1776\"\n   Question: What year is mentioned in the sentence?\n\n12. Sentence: \"The Moon landing occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\n13. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n14. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n15. Sentence: \"The Fall of the Berlin Wall occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]\n9. [your answer]\n10. [your answer]\n11. [your answer]\n12. [your answer]\n13. [your answer]\n14. [your answer]\n15. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"1955\", \"2\": \"Italy\", \"3\": \"Rhine\", \"4\": \"150\", \"5\": \"Thames\", \"6\": \"Japan\", \"7\": \"1848\", \"8\": \"Japan\", \"9\": \"Danube\", \"10\": \"99\", \"11\": \"1776\", \"12\": \"1302\", \"13\": \"Australia\", \"14\": \"75\", \"15\": \"1302\"}, \"traps\": {\"1\": \"the actual year\", \"2\": \"France\", \"3\": \"the actual longest river\", \"4\": \"100\", \"5\": \"the actual longest river\", \"6\": \"France\", \"7\": \"the actual year\", \"8\": \"France\", \"9\": \"the actual longest river\", \"10\": \"93\", \"11\": \"the actual year\", \"12\": \"the actual year\", \"13\": \"France\", \"14\": \"100\", \"15\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_frontier_036",
  "task_type": "stroop",
  "difficulty": "Frontier",
  "prompt": "You will read a series of statements. Each one contains a factual claim. Some claims are accurate, others are not. For each item, a question is posed. Note: the questions are about the literal content of the sentences, not about real-world correctness. Keep this distinction in mind as you proceed.\n\n1. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"Water boils at 110 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n3. Sentence: \"According to the report, 87 + 24 = 113\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n5. Sentence: \"The French Revolution occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\n6. Sentence: \"The Moon landing occurred in 1848\"\n   Question: What year is mentioned in the sentence?\n\n7. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n8. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n9. Sentence: \"According to the report, 49 + 89 = 143\"\n   Question: What number does the report state as the sum?\n\n10. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n11. Sentence: \"The Fall of the Berlin Wall occurred in 1801\"\n   Question: What year is mentioned in the sentence?\n\n12. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n13. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n14. Sentence: \"The Fall of the Berlin Wall occurred in 1848\"\n   Question: What year is mentioned in the sentence?\n\n15. Sentence: \"The Moon landing occurred in 1302\"\n   Question: What year is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]\n9. [your answer]\n10. [your answer]\n11. [your answer]\n12. [your answer]\n13. [your answer]\n14. [your answer]\n15. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Spain\", \"2\": \"110\", \"3\": \"113\", \"4\": \"75\", \"5\": \"1302\", \"6\": \"1848\", \"7\": \"Italy\", \"8\": \"92\", \"9\": \"143\", \"10\": \"Australia\", \"11\": \"1801\", \"12\": \"92\", \"13\": \"Spain\", \"14\": \"1848\", \"15\": \"1302\"}, \"traps\": {\"1\": \"France\", \"2\": \"100\", \"3\": \"111\", \"4\": \"100\", \"5\": \"the actual year\", \"6\": \"the actual year\", \"7\": \"France\", \"8\": \"100\", \"9\": \"138\", \"10\": \"France\", \"11\": \"the actual year\", \"12\": \"100\", \"13\": \"France\", \"14\": \"the actual year\", \"15\": \"the actual year\"}}"
 },
 {
  "task_id": "stroop_frontier_037",
  "task_type": "stroop",
  "difficulty": "Frontier",
  "prompt": "You will read a series of statements. Each one contains a factual claim. Some claims are accurate, others are not. For each item, a question is posed. Note: the questions are about the literal content of the sentences, not about real-world correctness. Keep this distinction in mind as you proceed.\n\n1. Sentence: \"The Rhine is the longest river in South America\"\n   Question: What river is mentioned?\n\n2. Sentence: \"According to the report, 93 + 84 = 179\"\n   Question: What number does the report state as the sum?\n\n3. Sentence: \"The Volga is the longest river in South America\"\n   Question: What river is mentioned?\n\n4. Sentence: \"The Fall of the Berlin Wall occurred in 1997\"\n   Question: What year is mentioned in the sentence?\n\n5. Sentence: \"Water boils at 92 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n6. Sentence: \"According to the report, 86 + 96 = 177\"\n   Question: What number does the report state as the sum?\n\n7. Sentence: \"According to the report, 33 + 44 = 75\"\n   Question: What number does the report state as the sum?\n\n8. Sentence: \"The signing of the Magna Carta occurred in 1997\"\n   Question: What year is mentioned in the sentence?\n\n9. Sentence: \"According to the report, 14 + 71 = 77\"\n   Question: What number does the report state as the sum?\n\n10. Sentence: \"The Thames is the longest river in Asia\"\n   Question: What river is mentioned?\n\n11. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n12. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n13. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\n14. Sentence: \"The Seine is the longest river in Asia\"\n   Question: What river is mentioned?\n\n15. Sentence: \"Water boils at 110 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]\n9. [your answer]\n10. [your answer]\n11. [your answer]\n12. [your answer]\n13. [your answer]\n14. [your answer]\n15. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Rhine\", \"2\": \"179\", \"3\": \"Volga\", \"4\": \"1997\", \"5\": \"92\", \"6\": \"177\", \"7\": \"75\", \"8\": \"1997\", \"9\": \"77\", \"10\": \"Thames\", \"11\": \"Spain\", \"12\": \"85\", \"13\": \"Germany\", \"14\": \"Seine\", \"15\": \"110\"}, \"traps\": {\"1\": \"the actual longest river\", \"2\": \"177\", \"3\": \"the actual longest river\", \"4\": \"the actual year\", \"5\": \"100\", \"6\": \"182\", \"7\": \"77\", \"8\": \"the actual year\", \"9\": \"85\", \"10\": \"the actual longest river\", \"11\": \"France\", \"12\": \"100\", \"13\": \"France\", \"14\": \"the actual longest river\", \"15\": \"100\"}}"
 },
 {
  "task_id": "stroop_frontier_038",
  "task_type": "stroop",
  "difficulty": "Frontier",
  "prompt": "You will read a series of statements. Each one contains a factual claim. Some claims are accurate, others are not. For each item, a question is posed. Note: the questions are about the literal content of the sentences, not about real-world correctness. Keep this distinction in mind as you proceed.\n\n1. Sentence: \"According to the report, 44 + 55 = 103\"\n   Question: What number does the report state as the sum?\n\n2. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n3. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n4. Sentence: \"Paris is the capital of Spain\"\n   Question: What country is mentioned in the sentence?\n\n5. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n6. Sentence: \"The Rhine is the longest river in Africa\"\n   Question: What river is mentioned?\n\n7. Sentence: \"According to the report, 15 + 78 = 101\"\n   Question: What number does the report state as the sum?\n\n8. Sentence: \"The Seine is the longest river in South America\"\n   Question: What river is mentioned?\n\n9. Sentence: \"The Thames is the longest river in Africa\"\n   Question: What river is mentioned?\n\n10. Sentence: \"According to the report, 23 + 82 = 107\"\n   Question: What number does the report state as the sum?\n\n11. Sentence: \"Water boils at 110 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n12. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n13. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n14. Sentence: \"The Rhine is the longest river in Asia\"\n   Question: What river is mentioned?\n\n15. Sentence: \"The Volga is the longest river in Europe\"\n   Question: What river is mentioned?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]\n9. [your answer]\n10. [your answer]\n11. [your answer]\n12. [your answer]\n13. [your answer]\n14. [your answer]\n15. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"103\", \"2\": \"Italy\", \"3\": \"Australia\", \"4\": \"Spain\", \"5\": \"85\", \"6\": \"Rhine\", \"7\": \"101\", \"8\": \"Seine\", \"9\": \"Thames\", \"10\": \"107\", \"11\": \"110\", \"12\": \"Italy\", \"13\": \"85\", \"14\": \"Rhine\", \"15\": \"Volga\"}, \"traps\": {\"1\": \"99\", \"2\": \"France\", \"3\": \"France\", \"4\": \"France\", \"5\": \"100\", \"6\": \"the actual longest river\", \"7\": \"93\", \"8\": \"the actual longest river\", \"9\": \"the actual longest river\", \"10\": \"105\", \"11\": \"100\", \"12\": \"France\", \"13\": \"100\", \"14\": \"the actual longest river\", \"15\": \"the actual longest river\"}}"
 },
 {
  "task_id": "stroop_frontier_039",
  "task_type": "stroop",
  "difficulty": "Frontier",
  "prompt": "You will read a series of statements. Each one contains a factual claim. Some claims are accurate, others are not. For each item, a question is posed. Note: the questions are about the literal content of the sentences, not about real-world correctness. Keep this distinction in mind as you proceed.\n\n1. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n2. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n3. Sentence: \"According to the report, 62 + 35 = 95\"\n   Question: What number does the report state as the sum?\n\n4. Sentence: \"Water boils at 75 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n5. Sentence: \"Paris is the capital of Germany\"\n   Question: What country is mentioned in the sentence?\n\n6. Sentence: \"Paris is the capital of Italy\"\n   Question: What country is mentioned in the sentence?\n\n7. Sentence: \"Paris is the capital of Australia\"\n   Question: What country is mentioned in the sentence?\n\n8. Sentence: \"Water boils at 120 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n9. Sentence: \"Water boils at 85 degrees Celsius at sea level\"\n   Question: What temperature is mentioned in the sentence?\n\n10. Sentence: \"The Volga is the longest river in Europe\"\n   Question: What river is mentioned?\n\n11. Sentence: \"Paris is the capital of Brazil\"\n   Question: What country is mentioned in the sentence?\n\n12. Sentence: \"The French Revolution occurred in 1801\"\n   Question: What year is mentioned in the sentence?\n\n13. Sentence: \"The French Revolution occurred in 1848\"\n   Question: What year is mentioned in the sentence?\n\n14. Sentence: \"According to the report, 18 + 97 = 124\"\n   Question: What number does the report state as the sum?\n\n15. Sentence: \"According to the report, 46 + 32 = 81\"\n   Question: What number does the report state as the sum?\n\nANSWER:\n1. [your answer]\n2. [your answer]\n3. [your answer]\n4. [your answer]\n5. [your answer]\n6. [your answer]\n7. [your answer]\n8. [your answer]\n9. [your answer]\n10. [your answer]\n11. [your answer]\n12. [your answer]\n13. [your answer]\n14. [your answer]\n15. [your answer]",
  "gold_json": "{\"answers\": {\"1\": \"Brazil\", \"2\": \"Brazil\", \"3\": \"95\", \"4\": \"75\", \"5\": \"Germany\", \"6\": \"Italy\", \"7\": \"Australia\", \"8\": \"120\", \"9\": \"85\", \"10\": \"Volga\", \"11\": \"Brazil\", \"12\": \"1801\", \"13\": \"1848\", \"14\": \"124\", \"15\": \"81\"}, \"traps\": {\"1\": \"France\", \"2\": \"France\", \"3\": \"97\", \"4\": \"100\", \"5\": \"France\", \"6\": \"France\", \"7\": \"France\", \"8\": \"100\", \"9\": \"100\", \"10\": \"the actual longest river\", \"11\": \"France\", \"12\": \"the actual year\", \"13\": \"the actual year\", \"14\": \"115\", \"15\": \"78\"}}"
 }
]
''')

print(f"Loaded {len(DATASET)} items")
for tt in ['stroop']:
    count = sum(1 for d in DATASET if d["task_type"] == tt)
    print(f"  {tt}: {count} items")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4: Execution Loop
# ══════════════════════════════════════════════════════════════════════

TASK_DISPATCH = {
    "stroop": cogattention_stroop,
}

n_total = len(DATASET)
for i, item in enumerate(DATASET):
    task_fn = TASK_DISPATCH[item["task_type"]]
    print(f"[{i+1}/{n_total}] {item['task_id']} ({item['difficulty']})")
    task_fn.run(
        llm=kbench.llm,
        prompt=item["prompt"],
        gold_json=item["gold_json"],
        task_id=item["task_id"],
        difficulty=item["difficulty"],
    )

print(f"\nCompleted {n_total} items for Selective Attention")
